In [2]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,ID,text,label
0,1,Atât e de înfricoșat de gândești că e turbat a...,graiul moldovenesc
1,3,Cei ce n-au dreptate n-ar mai năzui în veci la...,graiul moldovenesc
2,5,Deoarece ați primit bani de la oaspetele dumne...,graiul moldovenesc
3,7,Primiți vă rog oameni buni această mică mulțăm...,graiul moldovenesc
4,9,Fiin'c-o vint la noi vezi binie baș în zâua dă...,graiul bănățean


In [4]:
import re
suma = 0
for _, row in df_train.iterrows():
    cuvinte = re.findall(r'\b\w+\b', row.text.lower())
    suma += cuvinte.count('pâni')

for _, row in df_test.iterrows():
    cuvinte = re.findall(r'\b\w+\b', row.text.lower())
    suma += cuvinte.count('pâni')

subtask1 = suma

import string
def get_punct_marks(text):
    count = sum([1 for char in text if char in string.punctuation])
    return count

df_train["nr_punct"] = df_train.text.apply(get_punct_marks)
dicty = pd.DataFrame(df_train.groupby(["label"]).nr_punct.mean())
diff = abs(dicty.loc["graiul moldovenesc"] - dicty.loc["graiul bănățean"])
subtask2 = round(diff.iloc[0], 2)

df_1 = pd.DataFrame({
    "subtaskID": [1, 2],
    "datapointID": 1,
    "answer": [str(subtask1), str(subtask2)]
})

df_1

,subtaskID,datapointID,answer
0,1,1,13
1,2,1,0.05


In [5]:
diacritice = ["ă", "â", "î", "ș", "ț", "Ă", "Â", "Î", "Ș", "Ț"]
def get_diacritice(text):
    count = sum(1 for char in text if char in diacritice)
    return count

subtask3 = df_test.text.apply(get_diacritice)

df_2 = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.ID,
    "answer": subtask3
})

df_2.head()

,subtaskID,datapointID,answer
0,3,1491,2
1,3,1493,3
2,3,1495,3
3,3,1497,5
4,3,1499,8


In [6]:
X = df_train["text"]
y = df_train["label"]

X_final = df_test["text"]

from sklearn.feature_extraction.text import TfidfVectorizer
vect = TfidfVectorizer(analyzer='char', ngram_range=((3, 5)))
X = vect.fit_transform(X)
X_final = vect.transform(X_final)

X

<424x25449 sparse matrix of type '<class 'numpy.float64'>'
	with 73122 stored elements in Compressed Sparse Row format>

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

model = LinearSVC()

cross_val_score(model, X, y, cv=5, scoring='f1_macro').mean()

0.9563886243890354

In [8]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [9]:
df_3 = pd.DataFrame({
    "subtaskID": 4,
    "datapointID": df_test.ID,
    "answer": y_preds
})

df_3.head()

,subtaskID,datapointID,answer
0,4,1491,graiul moldovenesc
1,4,1493,româna standard
2,4,1495,româna standard
3,4,1497,graiul moldovenesc
4,4,1499,graiul moldovenesc


In [10]:
submission = pd.concat([df_1, df_2, df_3])
submission

,subtaskID,datapointID,answer
0,1,1,13
1,2,1,0.05
0,3,1491,2
1,3,1493,3
2,3,1495,3
...,...,...,...
102,4,1695,româna standard
103,4,1697,româna standard
104,4,1699,graiul bănățean
105,4,1701,româna standard


In [11]:
submission.to_csv("submission.csv", index=False)